In [1]:
import pandas as pd
import os

In [ ]:
def create_soccer_datasets(raw_csv_path):
    # 1. loading in the the raw dataset values
    try:
        df = pd.read_csv(raw_csv_path, encoding='utf-8-sig')
    except FileNotFoundError:
        print(f"Error: Could not find {raw_csv_path}. Please download it first.")
        return

    # Create the output directory matching your scaffolding
    output_dir = 'data/season_24-25'
    os.makedirs(output_dir, exist_ok=True)

    # 2. Anchor columns
    # so that its possible to join tables (merging Attacking and Possession)
    anchor_cols = ['Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', '90s']

    # 3. Define the specific columns for the 5 Agent datasets
    # Standard (Great for quick overviews and xG)
    standard_cols = anchor_cols + ['Gls', 'Ast', 'G+A', 'xG', 'npxG', 'xAG', 'PrgC', 'PrgP', 'PrgR']
    
    # Passing (Crucial for Offensive Burden metrics) and midfielders
    passing_cols = anchor_cols + ['PasTotCmp', 'PasTotAtt', 'PasTotCmp%', 'PasTotDist', 'PasPrgDist', 'Ast', 'xAG', 'xA', 'KP', 'Pas3rd', 'PPA', 'CrsPA', 'PasProg']
    
    # Dataset 3: Defense
    defense_cols = anchor_cols + ['Tkl', 'TklWon', 'TklDef3rd', 'TklMid3rd', 'TklAtt3rd', 'Blocks', 'Int', 'Tkl+Int', 'Clr', 'Err']
    
    # Dataset 4: Shooting (all players minus goalkeepers)
    shooting_cols = anchor_cols + ['Gls', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'Dist', 'FK', 'xG', 'npxG', 'npxG/Sh']
    
    # Dataset 5: Possession (The core of your Usage Rate metric)
    possession_cols = anchor_cols + ['Touches', 'TouDefPen', 'TouDef3rd', 'TouMid3rd', 'TouAtt3rd', 'TouAttPen', 'TouLive', 'ToAtt', 'ToSuc', 'ToSuc%', 'Carries', 'CarTotDist', 'CarPrgDist', 'CarProg', 'Car3rd', 'CPA', 'CarMis', 'CarDis', 'Rec']

    # 4. Map and Export
    datasets = {
        'top_5_leagues_standard.csv': standard_cols,
        'top_5_leagues_passing.csv': passing_cols,
        'top_5_leagues_defense.csv': defense_cols,
        'top_5_leagues_shooting.csv': shooting_cols,
        'top_5_leagues_possession.csv': possession_cols
    }

    for filename, columns in datasets.items():
        # Keep only the columns that actually exist in the raw CSV to prevent KeyError
        valid_cols = [col for col in columns if col in df.columns]
        
        # Create subset and drop players with zero minutes to keep data clean
        subset_df = df[valid_cols].copy()
        subset_df = subset_df[subset_df['90s'] > 0] 
        
        output_path = os.path.join(output_dir, filename)
        subset_df.to_csv(output_path, index=False)
        print()
        print(f"Created {filename} with {len(valid_cols)} columns and {len(subset_df)} rows.")

In [6]:
create_soccer_datasets('../data/historical/raw_kaggledata.csv')

Created top_5_leagues_standard.csv with 16 columns and 2805 rows.
Created top_5_leagues_passing.csv with 13 columns and 2805 rows.
Created top_5_leagues_defense.csv with 13 columns and 2805 rows.
Created top_5_leagues_shooting.csv with 20 columns and 2805 rows.
Created top_5_leagues_possession.csv with 11 columns and 2805 rows.
